# Summary

This notebook performs a raw extract from Azure Cosmos DB change feed into Unity Catalog volumes using Structured Streaming.

- Collects runtime parameters from widgets
- Builds Cosmos DB read configuration using service principal credentials from secrets
- Creates an audit manifest table if needed
- Writes each micro-batch as JSON Lines to a deterministic landing path
- Records batch metadata for idempotent tracking and replay safety
- Runs an `availableNow` streaming extract from the Cosmos change feed
- Verifies written files and manifest output at the end


In [0]:
import json, re
from datetime import datetime, timezone
from pyspark.sql import functions as F

defaults = {
    "tenant_id": "", "subscription_id": "", "resource_group": "rg-cosmos-dbx-poc",
    "cosmos_endpoint": "https://cosmos-dbx-poc-sm.documents.azure.com:443/",
    "cosmos_database": "pocdb", "cosmos_container": "customers",
    "start_from": "Beginning",          # first run only: Beginning | Now | ISO-8601 timestamp
    "max_items_per_batch": "100000",
    "uc_catalog": "cosmos_poc", "secret_scope": "kv-poc",
    "run_id": "manual",                 # job passes {{job.run_id}}
    "client_id_secret": "sp-cosmos-client-id",
    "client_secret_secret": "sp-cosmos-client-secret",
}
for k, v in defaults.items():
    dbutils.widgets.text(k, v)
p = {k: dbutils.widgets.get(k).strip() for k in defaults}

missing = [k for k in ("tenant_id", "subscription_id", "cosmos_endpoint") if not p[k]]
if missing:
    raise ValueError(f"Missing required parameters: {missing}")

safe = lambda s: re.sub(r"[^A-Za-z0-9_-]", "_", s)
SOURCE_KEY   = f"{safe(p['cosmos_database'])}/{safe(p['cosmos_container'])}"
LANDING_ROOT = f"/Volumes/{p['uc_catalog']}/raw/cosmos_landing/{SOURCE_KEY}"
CHECKPOINT   = f"/Volumes/{p['uc_catalog']}/ops/checkpoints/extract/{SOURCE_KEY}"
MANIFEST     = f"{p['uc_catalog']}.ops.extract_manifest"
print(f"Source: {SOURCE_KEY}\nLanding: {LANDING_ROOT}\nCheckpoint: {CHECKPOINT}")

dbutils.fs.mkdirs(LANDING_ROOT)   # bronze must never fail on a missing path for a new/empty source

In [0]:
cosmos_cfg = {
    "spark.cosmos.accountEndpoint": p["cosmos_endpoint"],
    "spark.cosmos.auth.type": "ServicePrincipal",
    "spark.cosmos.account.tenantId": p["tenant_id"],
    "spark.cosmos.account.subscriptionId": p["subscription_id"],
    "spark.cosmos.account.resourceGroupName": p["resource_group"],
    "spark.cosmos.auth.aad.clientId": dbutils.secrets.get(p["secret_scope"], p["client_id_secret"]),
    "spark.cosmos.auth.aad.clientSecret": dbutils.secrets.get(p["secret_scope"], p["client_secret_secret"]),
    "spark.cosmos.database": p["cosmos_database"],
    "spark.cosmos.container": p["cosmos_container"],
    "spark.cosmos.read.inferSchema.enabled": "false",            # _rawBody = original JSON
    "spark.cosmos.changeFeed.mode": "LatestVersion",
    "spark.cosmos.changeFeed.startFrom": p["start_from"],
    "spark.cosmos.changeFeed.itemCountPerTriggerHint": p["max_items_per_batch"],
    "spark.cosmos.applicationName": "dbx-raw-extract",
}

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MANIFEST} (
  source_key STRING, batch_id BIGINT, run_id STRING, extracted_at TIMESTAMP,
  doc_count BIGINT, min_ts BIGINT, max_ts BIGINT, landing_path STRING)
COMMENT 'One row per extract micro-batch - audit trail for the raw layer'
""")

In [0]:
MANIFEST_SCHEMA = ("source_key STRING, batch_id BIGINT, run_id STRING, extracted_at TIMESTAMP, "
                   "doc_count BIGINT, min_ts BIGINT, max_ts BIGINT, landing_path STRING")
run_batches = []

def write_raw(batch_df, batch_id):
    ss = batch_df.sparkSession
    batch_df = batch_df.persist()
    try:
        s = batch_df.agg(F.count("*").alias("n"), F.min("_ts").alias("lo"), F.max("_ts").alias("hi")).first()
        if s["n"] == 0:
            return
        target = f"{LANDING_ROOT}/batch_id={batch_id:010d}"     # deterministic -> retries overwrite
        (batch_df
            .select(F.regexp_replace("_rawBody", r"[\r\n]+", " ").alias("value"))  # guarantee JSON Lines
            .write.mode("overwrite")
            .option("maxRecordsPerFile", 100000)
            .text(target))
        ss.createDataFrame(
            [(SOURCE_KEY, batch_id, p["run_id"], datetime.now(timezone.utc), s["n"], s["lo"], s["hi"], target)],
            MANIFEST_SCHEMA).createOrReplaceTempView("_manifest_row")
        ss.sql(f"""MERGE INTO {MANIFEST} t USING _manifest_row s
                   ON t.source_key = s.source_key AND t.batch_id = s.batch_id
                   WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
        run_batches.append((batch_id, s["n"]))
    finally:
        batch_df.unpersist()

In [0]:
run_batches.clear()
stream = spark.readStream.format("cosmos.oltp.changeFeed").options(**cosmos_cfg).load()

if not {"_rawBody", "_ts"}.issubset(stream.columns):
    raise RuntimeError(f"Unexpected change feed schema: {stream.columns}")

q = (stream.writeStream
        .queryName(f"extract_{safe(p['cosmos_container'])}")
        .foreachBatch(write_raw)
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start())
q.awaitTermination()

result = {"source": SOURCE_KEY, "run_id": p["run_id"],
          "batches": len(run_batches), "docs": sum(n for _, n in run_batches)}
print(json.dumps(result))

In [0]:
display(spark.table(MANIFEST).orderBy("batch_id"))
print("Raw lines on disk:", spark.read.text(f"{LANDING_ROOT}/*/*.txt").count())